# Dataset Loading 

In [5]:
%load_ext noworkflow

import pandas as pd
import numpy as np
dataframe = pd.read_csv("datasets/heart.csv")
dataframe.head()

The noworkflow extension is already loaded. To reload it, use:
  %reload_ext noworkflow


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## MLflow Tracking

In [6]:
from pathlib import Path

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

mlflow_tracking_dir = Path("mlruns").resolve()
mlflow.set_tracking_uri(mlflow_tracking_dir.as_uri())
mlflow.set_experiment("heart-disease-svm")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

# Data Exploration and Analysis

In [ ]:
dataframe.describe()

In [ ]:
dataframe.info()

In [ ]:
# Setting 6th row, cholesterol as NULL, to simulate a dataset having a row with null value
dataframe.loc[5, 'Cholesterol'] = np.nan

# Checking null values in a dataset
dataframe.isnull().sum()

In [ ]:
# Printing 4th to 8th rows
dataframe.loc[[3,4,5,6,7]]

In [ ]:
# Dropping null rows (in this case, 6th row. Notice that now total rows is 917)
# inplace=True used to make the change in the dataframe itself, and not return a copy instead
dataframe.dropna(inplace = True) 

In [ ]:
dataframe.head(7)

## Plotting

In [ ]:
import matplotlib.pyplot as plt

dataframe.plot(kind='scatter', x='Cholesterol' , y='RestingBP', color='red')

***But wait, according to the scatterplot, there are rows with cholesterol 0. This is not realistic, and must be a problem with the dataset. Therefore, all rows with cholesterol 0 must be dropped.***

In [ ]:
for x in dataframe.index:
  if dataframe.loc[x, "Cholesterol"] == 0:
    dataframe.drop(x, inplace = True)
    print(f"Row {x} dropped")

In [ ]:
dataframe.info()

## Converting Categorical Data to Numerical Data (One Hot Encoding)

In [ ]:
dataframe.head(10)

In [ ]:
# One-Hot Encoding
temp_df = pd.get_dummies(dataframe["Sex"], dtype=int)
temp_df.head()

In [ ]:
temp_df.drop(columns=["F"],inplace=True)
dataframe.drop(columns=["Sex"], inplace=True)
dataframe = pd.concat([dataframe,temp_df], axis=1)
dataframe.rename(columns={"M":"Sex"}, inplace=True)
dataframe.head()

## Converting Categorical Data to Numerical Data (using Label Encoder)

**Some features have more than two categories. These categories must be replaced with 0,1,2,3... before passing to model. To achieve this, sklearn's LabelEncoder is used to replace categorical values with numerical values. The mapping is shown for each replacement.**

In [ ]:
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

dataframe['ChestPainTypeLE'] = LE.fit_transform(dataframe['ChestPainType'])
mapping = dict(zip(LE.classes_, LE.transform(LE.classes_)))
print(mapping)

In [ ]:
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

dataframe['RestingECGLE'] = LE.fit_transform(dataframe['RestingECG'])
mapping = dict(zip(LE.classes_, LE.transform(LE.classes_)))
print(mapping)

In [ ]:
# Note that this can be done with One-Hot Encoding as well
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

dataframe['ExerciseAnginaLE'] = LE.fit_transform(dataframe['ExerciseAngina'])
mapping = dict(zip(LE.classes_, LE.transform(LE.classes_)))
print(mapping)

In [ ]:
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

dataframe['ST_SlopeLE'] = LE.fit_transform(dataframe['ST_Slope'])
mapping = dict(zip(LE.classes_, LE.transform(LE.classes_)))
print(mapping)

In [ ]:
dataframe.head()

In [ ]:
dataframe.drop(columns=["ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope" ] , inplace=True)
dataframe.head()

In [ ]:
import seaborn as sns

corr_matrix = dataframe.corr()

sns.heatmap(corr_matrix, cmap='Purples', annot=True)

plt.show()

# Train-Test Splitting

In [ ]:
from sklearn.model_selection import train_test_split

X = dataframe.drop(['HeartDisease'], axis=1)

y = dataframe['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=0)

# Model Training

## SVM

In [ ]:
from sklearn.svm import SVC

svm = SVC(random_state=42)

svm.fit(X_train,y_train)
y_predicted_svm = svm.predict(X_test)
print(y_predicted_svm)

## Decision Tree

## Naive Bayes

## Random Forest 

# Prediction

In [ ]:
feature_cols = [
    "Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak", "Sex", "ChestPainTypeLE", 
    "RestingECGLE", "ExerciseAnginaLE", "ST_SlopeLE",
]

## SVM

In [ ]:
example1 = {
    "Age"               : 40.0,
    "RestingBP"         : 140.0,
    "Cholesterol"       : 289.0,
    "FastingBS"         :   0.0,
    "MaxHR"             : 172.0,
    "Oldpeak"           :   0.0,
    "Sex"               :   1.0,
    "ChestPainTypeLE"   :   1.0,
    "RestingECGLE"      :   1.0,
    "ExerciseAnginaLE"  :   0.0,
    "ST_SlopeLE"        :   2.0,
}

df_example1 = pd.DataFrame([example1], columns=feature_cols)

prediction1 = svm.predict(df_example1)
print(f"The predicted class by SVM is {prediction1[0]}")

In [ ]:
example2 = {
    "Age"               : 54.0,
    "RestingBP"         : 125.0,
    "Cholesterol"       : 224.0,
    "FastingBS"         :   0.0,
    "MaxHR"             : 122.0,
    "Oldpeak"           :   2.0,
    "Sex"               :   1.0,
    "ChestPainTypeLE"   :   0.0,
    "RestingECGLE"      :   1.0,
    "ExerciseAnginaLE"  :   0.0,
    "ST_SlopeLE"        :   1.0,
}

df_example2 = pd.DataFrame([example2], columns=feature_cols)

prediction2 = svm.predict(df_example2)
print(f"The predicted class by SVM is {prediction2[0]}")

## Decision Tree

## Naive Bayes

## Random Forest

# Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = {
    "accuracy": accuracy_score(y_test, y_predicted_svm),
    "precision_macro": precision_score(y_test, y_predicted_svm, average='macro'),
    "recall_macro": recall_score(y_test, y_predicted_svm, average='macro'),
    "f1_macro": f1_score(y_test, y_predicted_svm, average='macro'),
}

print("------------------------------")
print(f"Accuracy : {metrics['accuracy']}")
print(f"Precision: {metrics['precision_macro']}")
print(f"Recall   : {metrics['recall_macro']}")
print(f"F1 Score : {metrics['f1_macro']}")
print("------------------------------")

run_params = {
    "model_type": "SVC",
    "dataset": "heart.csv",
    "test_size": 0.2,
    "split_random_state": 0,
    "model_random_state": svm.get_params().get("random_state"),
    "rows_after_cleaning": len(dataframe),
    "feature_count": len(feature_cols),
}

signature = infer_signature(X_test, y_predicted_svm)
input_example = X_test.head(5)

with mlflow.start_run(run_name="svm_baseline") as run:
    mlflow.log_params(run_params)
    mlflow.log_metrics(metrics)
    mlflow.log_dict({"feature_columns": feature_cols}, "feature_columns.json")
    mlflow.sklearn.log_model(
        sk_model=svm,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
    )

print(f"MLflow run_id: {run.info.run_id}")

## Decision Tree

## Naive Bayes

## Random Forest